## Statistics of Lyman-α profiles, metal emission lines and absorption lines

This notebook generates histograms and scatter plots of key parameters in the megatable, and searches for relationships between them

In [ ]:
from astropy.io import fits
import astropy.table as aptb

from tangelo.statistics import check_line_correlations, check_lya_correlations

# Load the megatab
SPEC_SOURCE = "R21"  #  Where are the spectra from? 'R21' for Richard+21, 'APER' for apertures
SPEC_TYPE   = "weight_skysub"

tabfile = f"megatables/fit_catalog_qc_cpts_stack_zeldamcmc_refit_sysv_absv_{SPEC_TYPE}.fits"
tabhdul = fits.open(tabfile)
megatable = aptb.Table(tabhdul[1].data)

# Only keep sources with significant Lyman alpha detection
megatab = megatable[(megatable['SNRR'] > 5.0) | (megatable['SNRB'] > 5.0)]

In [ ]:
lya_properties = [
    "LYA_EW",
    "DISPR",
    "LUM_CONT_LYA",
    "LUM_LYA",
    "ASYMR",
    "FWHMR",
    "BRRATIO",
    "BRSEP",
    "DELTAV_LYA",
    "VEXP_ZELDA",
    "TDUST_ZELDA",
    "LOGIEW_ZELDA",
    "WINT_ZELDA",
    "LOGN_ZELDA",
]

lya_summaries = check_lya_correlations(lya_properties, lya_properties, megatab, min_points=6,
                                       significance_thresh=0.05, mcmc=True, logify=True, c='SNRR',
                                       save_fig=True, point_sig_thresh=5.0, clip_extreme_errors=0.33,
                                       cnorm='log', cmap='viridis')

In [ ]:
# Plot fitted line rest-frame EWs against Lyman alpha properties
import importlib
import tangelo.statistics
importlib.reload(tangelo.statistics)
from tangelo.statistics import check_line_correlations, check_lya_correlations  # re-import any names you use directly

lines_to_plot = [
    "SiII1260",
    "CII1334",
    "SiIV1394",
    "SiIV1403",
    "CIV1548",
    "HeII1640",
    "OIII1660",
    "CIII1907"
]

lya_properties = [
    "LYA_EW",
    "DISPR",
    "LUM_CONT_LYA",
    "LUM_LYA",
    "ASYMR",
    "FWHMR",
    "BRRATIO",
    "BRSEP",
    "DELTAV_LYA",
    "VEXP_ZELDA",
    "LOGN_ZELDA",
    "LOGIEW_ZELDA",
    "WINT_ZELDA",
    "TDUST_ZELDA",
]

# list to specify which lines should be treated as absorption
abs_lines = [
    "SiII1260",
    "CII1334",
    "SiIV1394",
    "SiIV1403",
]

line_properties = ["EW", "FWHM", "LUM", "CONT_LUM"]

summaries = {}
for line_prop in line_properties:
    fit_upper_limits = line_prop in ["EW"]
    summaries[line_prop] = check_line_correlations(line_prop, lya_properties, lines_to_plot, abs_lines, megatab,
                                                min_points=6, significance_thresh=0.05, mcmc=True, 
                                                fit_upper_limits=False, upper_limit=3.0, logify=True,
                                                plot_upper_limits=False, save_fig=True, point_sig_thresh=4.0,
                                                clip_extreme_errors=1.0, cnorm='log', cmap='viridis', c='SNRR')


In [ ]:
# Check correlations between Lyman alpha properties and outflow properties

outflow_lines = [
    "LI_ABS",
    "HI_ABS",
]

outflow_properties = [
    "EW",
    "DV",
    "W"
]

lya_properties_outflow = [
    "LYA_EW",
    "DISPR",
    "CONT",
    "ASYMR",
    "FWHMR",
    "BRRATIO",
    "BRSEP",
    "DELTAV_LYA",
    "VEXP_ZELDA",
    "LOGN_ZELDA",
]

for line_prop in outflow_properties:
    outflow_summaries = check_line_correlations(line_prop, lya_properties_outflow, outflow_lines, 
                                                outflow_lines, megatab, min_points=6, significance_thresh=0.05, 
                                                mcmc=True, fit_upper_limits=False, upper_limit=3.0, logify=False,
                                                plot_upper_limits=False, save_fig=True, point_sig_thresh=5.0)

In [ ]:
# Print summary of significant correlations
for line_prop, line_summaries in summaries.items():
    print(f"\nSummary of correlations for {line_prop}:")
    for line, lya_summaries in line_summaries.items():
        for lya_prop, summary in lya_summaries.items():
            if summary['p_value'] < 0.05:
                print(f"{line} {line_prop} vs {lya_prop}: slope={summary['slope']:.5f}±{summary['slope_err']:.5f}, "
                    f"intercept={summary['intercept']:.5f}±{summary['intercept_err']:.5f}, "
                    f"p={summary['p_value']:.3e}, n={summary['n_points']}")
                
for line_prop, line_summaries in outflow_summaries.items():
    print(f"\nSummary of correlations for {line_prop}:")
    for line, lya_summaries in line_summaries.items():
        for lya_prop, summary in lya_summaries.items():
            if summary['p_value'] < 0.05:
                print(f"{line} {line_prop} vs {lya_prop}: slope={summary['slope']:.5f}±{summary['slope_err']:.5f}, "
                    f"intercept={summary['intercept']:.5f}±{summary['intercept_err']:.5f}, "
                    f"p={summary['p_value']:.3e}, n={summary['n_points']}")

In [ ]:
from typing import Union

def ks_test_lya_mc(line: str, lya_prop: str, megatab: aptb.Table, line_property: str="EW", 
                   abs_lines: list = abs_lines, threshold: Union[float, str] ='auto',
                   significance_thresh: float = 3.0) -> tuple[float, float]:
    """
    Perform a KS test comparing the distribution of a given lyman alpha property
    (e.g. DISPR, FWHMR, ASYMR) for sources with high vs low values of a given line property (e.g. EW).
    Handles cases of upper limits in the line property by including in the low-value group
    all sources with detections OR upper limits below the required threshold.

    Parameters
    ----------
    line : str
        The line to analyze (e.g., "SiII1260").
    lya_prop : str
        The Lyman alpha property to split on (e.g., "LYA_EW").
    megatab : aptb.Table
        The megatable containing the data.
    line_property : str, optional
        The line property to use for splitting the sample (e.g. "EW", "FWHM"), by default "EW".
    abs_lines : list, optional
        List of lines that should be treated as absorption, by default abs_lines defined above.
    threshold : float or str, optional
        The threshold to use for splitting the sample into high vs low Lyman alpha property groups. If 'auto', will use the median value of the Lyman alpha property for splitting, by default 'auto'.
    significance_thresh : float, optional
        The sigma threshold to use for including sources based on their SNR, by default 3.0.

    Returns
    -------
    tuple[float, float]
        The KS statistic and p-value for the test.
    """
    from tangelo.statistics import get_line_property, get_lya_property, prepare_scatter_mask

    # Get the line property column and the Lyman alpha property column
    line_col, line_err = get_line_property(megatab, line, line_property, abs=line in abs_lines)
    lya_col, lya_col_err = get_lya_property(megatab, lya_prop)

    # Prepare mask for valid data points
    mask = prepare_scatter_mask(megatab, line, line_col, line_property, lya_col, lya_prop,
                                               abs_lines, include_upper_limits=True)
    
    significance_mask = (megatab[f"SNR_{line}"].copy() > significance_thresh)  # Mask for significant detections
    if line in abs_lines:
        # For absorption lines, we want to flip the sign of the SNR threshold to identify significant absorptions
        significance_mask = (megatab[f"SNR_{line}"].copy() < -significance_thresh)
    
    detections = mask & significance_mask  # Significant detections
    upper_limits = mask & (~significance_mask)  # Non-detections or upper limits

    # Replace non-detections with 3 sigma upper bounds
    line_col[upper_limits] = 3 * line_err[upper_limits]

    # Split the data into two groups based on the median value of the Lyman alpha property or provided threshold
    if threshold == 'auto':
        median_line = np.nanmedian(line_col)
        print(f"Using median {line} {line_property} value of {median_line:.2f} for splitting groups.")
    else:
        median_line = threshold
    group_high = lya_col[detections & (line_col >= median_line)]
    group_low = lya_col[(detections & (line_col < median_line)) | (upper_limits & (line_col < median_line))]

    # Logify the quantities if they are in the log_quantities list to ensure that the KS test is comparing the same transformed distributions as the correlation analysis
    if lya_prop in log_quantities:
        group_high = np.log10(group_high)
        group_low = np.log10(group_low)

    # If either group is empty, we cannot perform the KS test
    if len(group_high) == 0 or len(group_low) == 0:
        print(f"Warning: One of the groups for {line} {line_property} vs {lya_prop} is empty. Cannot perform KS test.")
        return np.nan, np.nan

    # Perform KS test comparing the line property distributions of the two groups
    ks_statistic, p_value = stats.ks_2samp(group_high, group_low)

    if p_value < 0.05:
        print(f"Significant difference in {lya_prop} distribution for high vs low {line} {line_property} (KS p={p_value:.3e}).")
    else:
        print(f"No significant difference in {lya_prop} distribution for high vs low {line} {line_property} (KS p={p_value:.3e}).")
        return ks_statistic, p_value  # Skip plotting if not significant to avoid clutter

    print(f"KS test for {line} {line_property} vs {lya_prop}: KS statistic={ks_statistic:.3f}, p-value={p_value:.3e}")

    n_bins = 20
    bins = np.linspace(min(np.min(group_high), np.min(group_low)), max(np.max(group_high), np.max(group_low)), n_bins)

    # Plot comparative histograms of the Lyman alpha property for the two groups
    plt.figure(figsize=(6, 4))
    plt.hist(group_high, bins=bins, alpha=0.7, label=f"{line} {line_property} $\geq {median_line:.2f}$", 
             color='steelblue', edgecolor='black')
    plt.hist(group_low, bins=bins, alpha=0.7, label=f"{line} {line_property} $< {median_line:.2f}$", 
             color='salmon', edgecolor='black')
    plt.xlabel(tgplt.get_plot_name(lya_prop))
    plt.ylabel("Number of sources")
    plt.title(f"{line} {line_property} vs {lya_prop}\nKS p-value={p_value:.3e}")
    plt.legend()
    plt.savefig(f"plots/{line}_{line_property}_vs_{lya_prop}_KS_hist.png", dpi=300, bbox_inches='tight')
    plt.show()  

    return ks_statistic, p_value

In [ ]:
# Now we perform two-sample KS tests to compare the distributions of Lyman alpha properties for sources
# above and below emission line EW thresholds

megatab_highlyasnr = megatab[megatab['SNRR'] > 10]  # Subset of sources with highly significant Lyman alpha detections

ks_results = {}
for line in lines_to_plot:
    ks_results[line] = {}
    for lya_prop in lya_properties:
        ks_statistic, p_value = ks_test_lya_mc(line, lya_prop, megatab_highlyasnr, line_property="EW", abs_lines=abs_lines,
                                               threshold='auto')
        ks_results[line][lya_prop] = {
            'ks_statistic': ks_statistic,
            'p_value': p_value
        }

In [ ]:
# Make scatter plots comparing Lyman alpha properties

lya_properties

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.decomposition import FactorAnalysis
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge

# =============================================================================
# Feature matrix construction
# =============================================================================
# "Complete" features (every source has these — Lya always detected)
# get_lya_property applies rest-frame /(1+z) and LSF quadrature subtraction
lya_feature_names = [
    # 'LYA_EW',
    # 'FWHMR',
    'DISPR',
    'ASYMR',
    'BRSEP',
    # 'BRRATIO',
    # 'CONT',
]

# Sparse line EWs (sign-flipped for absorption lines so all should be positive)
line_feature_defs = {
    # 'SiII1260' : dict(col='SiII1260',  is_abs=True),
    # 'CII1334'  : dict(col='CII1334',   is_abs=True),
    # 'SiIV1394' : dict(col='SiIV1394',  is_abs=True),
    # 'SiIV1403' : dict(col='SiIV1403',  is_abs=True),
    'CIV1548'  : dict(col='CIV1548',   is_abs=False),
    'HeII1640' : dict(col='HeII1640',  is_abs=False),
    'OIII1660' : dict(col='OIII1660',  is_abs=False),
    'CIII1907' : dict(col='CIII1907',  is_abs=False),
}

SNR_DET_THRESH  =  3.0   # |SNR| > this => detection
SNR_UL_THRESH   =  1.0   # |SNR| < this => treat as upper limit (not just noise)
N_COMPONENTS    =  3     # Number of latent factors to extract

# =============================================================================
# Build arrays
# =============================================================================
N = len(megatab)
n_lya  = len(lya_feature_names)
n_line = len(line_feature_defs)
n_feat = n_lya + n_line

feature_names = lya_feature_names + list(line_feature_defs.keys())

X       = np.full((N, n_feat), np.nan)   # raw values (log-space where appropriate)
X_ul    = np.full((N, n_feat), np.nan)   # upper-limit values
is_obs  = np.zeros((N, n_feat), dtype=bool)   # True = detected
is_ul   = np.zeros((N, n_feat), dtype=bool)   # True = upper limit (not detected but constrained)

# --- Lya features (always observed) — rest-frame + LSF corrected via get_lya_property ---
for j, name in enumerate(lya_feature_names):
    vals, _ = get_lya_property(megatab, name, rest_frame=True, correct_inst=True)
    vals = np.array(vals, dtype=float)
    finite = np.isfinite(vals) & (vals > 0)
    X[finite, j]      = np.log10(vals[finite])
    is_obs[finite, j] = True

# --- Line EWs — rest-frame corrected via get_line_property ---
z = np.array(megatab['z'], dtype=float)
for j, (name, defn) in enumerate(line_feature_defs.items(), start=n_lya):
    col    = defn['col']
    is_abs = defn['is_abs']

    # get_line_property applies /(1+z) rest-frame correction and sign-flip for absorption
    ew, ew_err = get_line_property(megatab, col, 'EW', rest_frame=True, abs=is_abs)
    ew     = np.array(ew,     dtype=float)
    snr      = np.array(megatab[f'SNR_{col}'],      dtype=float)
    cont     = np.array(megatab[f'CONT_{col}'],     dtype=float)
    cont_err = np.array(megatab[f'CONT_ERR_{col}'], dtype=float)
    flux_err = np.array(megatab[f'FLUX_ERR_{col}'], dtype=float)
    try:
        flag = np.array(megatab[f'FLAG_{col}'])
        unflagged = (flag == '')
    except KeyError:
        unflagged = np.ones(N, dtype=bool)

    if is_abs:
        det_mask = snr < -SNR_DET_THRESH
        ul_mask  = (snr >= -SNR_DET_THRESH) & (snr <= SNR_UL_THRESH)
    else:
        det_mask = snr > SNR_DET_THRESH
        ul_mask  = (snr >= -SNR_UL_THRESH) & (snr <= SNR_DET_THRESH)

    cont_ok  = (cont / cont_err > 3.0) if np.any(cont_err > 0) else np.ones(N, dtype=bool)
    det_mask &= unflagged & cont_ok & (ew > 0)
    ul_mask  &= cont_ok

    # 3-sigma upper limit on rest-frame |EW| from the flux error
    ew_ul = 3 * np.abs(flux_err) / np.abs(cont) / (1 + z)

    X[det_mask, j]      = np.log10(ew[det_mask])
    is_obs[det_mask, j] = True

    # Upper-limit: set UL value; X stays NaN so imputer will fill it
    X_ul[ul_mask, j]  = np.log10(np.maximum(ew_ul[ul_mask], 1e-6))
    is_ul[ul_mask, j] = True

print("Data matrix built.")
print(f"  Sources: {N}")
print(f"  Features: {n_feat}  ({n_lya} Lya + {n_line} line EWs)")
for j, name in enumerate(feature_names):
    n_det = is_obs[:, j].sum()
    n_ul  = is_ul[:, j].sum()
    n_mis = N - n_det - n_ul
    print(f"  {name:<12}  det={n_det:>4}  UL={n_ul:>4}  missing={n_mis:>4}")

# =============================================================================
# Imputation: MICE anchored by the always-observed Lya features
# =============================================================================
# IterativeImputer uses chained regression to fill NaN.
# With Lya features always observed they anchor every imputation round.
# min/max_value must be scalars or 1-D arrays (not dicts) for older sklearn versions.
imputer = IterativeImputer(
    estimator=BayesianRidge(),
    max_iter=20,
    random_state=42,
    min_value=-3,   # sensible log-EW floor
    max_value=3,    # sensible log-EW ceiling
)
X_imp = imputer.fit_transform(X)

# Clamp upper-limit entries: if imputed value > log(UL), replace with log(UL)
ul_idx = np.where(is_ul & ~is_obs)
X_imp[ul_idx] = np.minimum(X_imp[ul_idx], X_ul[ul_idx])

print("\nImputation complete.")

# =============================================================================
# Factor Analysis (EM-based, more robust than PCA for heteroscedastic data)
# =============================================================================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imp)

fa = FactorAnalysis(n_components=N_COMPONENTS, random_state=42, max_iter=2000)
scores = fa.fit_transform(X_scaled)       # (N, n_components)
loadings = fa.components_.T               # (n_feat, n_components)

# Explained variance (approximated as variance of projected scores)
total_var     = np.sum(np.var(X_scaled, axis=0))
explained_var = np.var(scores, axis=0)
explained_frac = explained_var / total_var
print(f"\nApproximate explained variance per factor:")
for k in range(N_COMPONENTS):
    print(f"  Factor {k+1}: {100*explained_frac[k]:.1f}%")

# =============================================================================
# Plots
# =============================================================================

# --- 1: Loadings heatmap ---
fig, ax = plt.subplots(figsize=(8, 5), facecolor='w')
im = ax.imshow(loadings, aspect='auto', cmap='RdBu_r',
               vmin=-np.abs(loadings).max(), vmax=np.abs(loadings).max())
ax.set_xticks(range(N_COMPONENTS))
ax.set_xticklabels([f'Factor {k+1}' for k in range(N_COMPONENTS)])
ax.set_yticks(range(n_feat))
ax.set_yticklabels(feature_names)
ax.set_title('Factor loadings')
for i in range(n_feat):
    ax.axhline(i + 0.5, color='gray', linewidth=0.4, alpha=0.5)
ax.axhline(n_lya - 0.5, color='k', linewidth=1.5)   # separator between Lya and line EWs
plt.colorbar(im, ax=ax, label='Loading')
plt.tight_layout()
plt.show()
plt.close(fig)

# --- 2: Biplot: scores + loading arrows for factors 1 and 2 ---
lya_ew_col, _ = get_lya_property(megatab, 'LYA_EW', rest_frame=True, correct_inst=True)
lya_ew_col = np.array(lya_ew_col, dtype=float)
lya_ew_log = np.where((lya_ew_col > 0) & np.isfinite(lya_ew_col),
                      np.log10(lya_ew_col), np.nan)

fig, ax = plt.subplots(figsize=(8, 7), facecolor='w')
sc = ax.scatter(scores[:, 0], scores[:, 1],
                c=lya_ew_log, cmap='plasma', alpha=0.6, s=20, zorder=2)
plt.colorbar(sc, ax=ax, label=r'$\log_{10}$(Ly$\alpha$ EW)')

# Arrow scale: normalise so longest arrow fits inside the score cloud
scale = 0.4 * max(np.ptp(scores[:, 0]), np.ptp(scores[:, 1]))
arrow_len = np.sqrt(loadings[:, 0]**2 + loadings[:, 1]**2)
max_len   = arrow_len.max()
for i, name in enumerate(feature_names):
    dx = loadings[i, 0] / max_len * scale
    dy = loadings[i, 1] / max_len * scale
    color = 'royalblue' if i < n_lya else 'crimson'
    ax.annotate('', xy=(dx, dy), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color=color, lw=1.5))
    ax.text(dx * 1.12, dy * 1.12, name, fontsize=8, color=color, ha='center')

ax.axhline(0, color='k', linewidth=0.5, alpha=0.4)
ax.axvline(0, color='k', linewidth=0.5, alpha=0.4)
ax.set_xlabel(f'Factor 1 ({100*explained_frac[0]:.1f}\\% var.)')
ax.set_ylabel(f'Factor 2 ({100*explained_frac[1]:.1f}\\% var.)')
ax.set_title('Biplot  [blue = Ly$\\alpha$ features, red = line EWs]')
plt.tight_layout()
plt.show()
plt.close(fig)

# --- 3: Factor scores vs individual Lya properties ---
fwhmr_vals, _ = get_lya_property(megatab, 'FWHMR', rest_frame=True, correct_inst=True)
brratio_vals, _ = get_lya_property(megatab, 'BRRATIO')
fwhmr_arr   = np.array(fwhmr_vals,   dtype=float)
brratio_arr = np.array(brratio_vals, dtype=float)

lya_plot_cols = {
    'LYA_EW' : lya_ew_log,
    'FWHMR'  : np.where((fwhmr_arr > 0) & np.isfinite(fwhmr_arr), np.log10(fwhmr_arr), np.nan),
    'ASYMR'  : np.array(megatab['ASYMR'], dtype=float),
    'BRRATIO': np.where((brratio_arr > 0) & np.isfinite(brratio_arr),
                        np.log10(np.clip(brratio_arr, 1e-4, None)), np.nan),
}

from scipy.stats import spearmanr
n_lya_plot = len(lya_plot_cols)
fig, axes = plt.subplots(N_COMPONENTS, n_lya_plot,
                          figsize=(4 * n_lya_plot, 3.5 * N_COMPONENTS),
                          facecolor='w')
axes = np.atleast_2d(axes)

for k in range(N_COMPONENTS):
    for j, (prop_name, prop_vals) in enumerate(lya_plot_cols.items()):
        ax = axes[k, j]
        good = np.isfinite(prop_vals) & np.isfinite(scores[:, k])
        ax.scatter(prop_vals[good], scores[good, k], alpha=0.4, s=15, color='steelblue')
        if good.sum() > 4:
            rho, p = spearmanr(prop_vals[good], scores[good, k])
            ax.text(0.04, 0.95, f'$\\rho = {rho:.2f}, p = {p:.2g}$',
                    transform=ax.transAxes, va='top', fontsize=9,
                    bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
        ax.set_xlabel(prop_name, fontsize=9)
        if j == 0:
            ax.set_ylabel(f'Factor {k+1} score', fontsize=9)

plt.suptitle('Factor scores vs Lyman-alpha properties', y=1.01)
plt.tight_layout()
plt.show()
plt.close(fig)
